In [ ]:
import os
import re
import json
import time
import nltk
import pandas as pd
from datetime import datetime, timezone
from typing import List, Set
from typing_extensions import TypedDict
from google import genai
from google.genai import types
from dotenv import load_dotenv

from nltk.corpus import stopwords

load_dotenv()

client: genai.Client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))
MODEL: str = "gemini-3.1-flash-lite-preview"

BATCH_SIZE: int = 15
CACHE_TTL: str = "3600s"         # 1 h (el máximo); la caché se renueva automáticamente si expira
CACHE_SAFETY_MARGIN: int = 60    # Un minuto antes del vencimiento para renovar.

In [2]:
nltk.download('stopwords')
STOP_WORDS: Set[str] = set(stopwords.words('spanish'))

def clean_concept(concept: str):
    if concept == "" or concept is None:
        return ""
    if (not isinstance(concept, str)):
        return str(concept)

    concept = concept.lower()
    tabla = str.maketrans(f"áäéëíïóöúü", "aaeeiioouu")
    concept = concept.translate(tabla)
    concept = re.sub(r'\(\s*[\d,.]+\s*%?\s*\)', '', concept)
    concept = re.sub(r'\b\d+\s*[xX]\s*\d+\b', '', concept)
    concept = re.sub(r'\b\d+(?:[.,]\d+)?\s*%', '', concept)
    meses: list[str] = [
        "enero", "febrero", "marzo", "abril", "mayo", "junio",
        "julio", "agosto", "septiembre", "octubre", "noviembre", "diciembre"
    ]
    pattern_meses = r'\b(?:' + '|'.join(meses) + r')\b'
    concept = re.sub(pattern_meses, 'mes', concept)
    concept = re.sub(r'\b\d+(?:[.,]\d+)?\b', '', concept)
    concept = re.sub(r'\s*[,.]\s*', ' ', concept)
    caracteres_especiales: str = '–-#()[]{}/:_*+.,~°";$&´='+"'"
    tabla = str.maketrans(caracteres_especiales, " " * len(caracteres_especiales))
    concept = concept.translate(tabla)
    concept = re.sub(r'\d+$', '', concept)
    concept = re.sub(r'[0-9]', ' ', concept)
    concept = re.sub(r'\s+', ' ', concept).strip()
    tokens = concept.split()
    tokens = [word for word in tokens if word not in STOP_WORDS and len(word) > 2]
    return " ".join(tokens)

[nltk_data] Downloading package stopwords to /home/vscode/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
INPUT_FILE: str = "./resources/input/Conceptos PTESA.xlsx"
df_input: pd.DataFrame = pd.read_excel(INPUT_FILE)
df_input: pd.DataFrame = df_input['Concepto'].astype(str).str.split('|').explode().to_frame()
df_input.head()

,Concepto
0,COSTO DIRECTO DE OBRA
0,ADMINISTRACION (22%)
0,IMPROVISTOS (3%)
0,"UTILIDAD (4,2016806722%) MAS IVA 19%"
0,ESTUDIOS Y DISEÑOS MAS IVA 19%


In [4]:
df_concepto_cleared: pd.DataFrame = df_input.copy()
df_concepto_cleared["concepto_cleared"] = df_concepto_cleared["Concepto"].apply(clean_concept)
df_concepto_cleared: pd.DataFrame = df_concepto_cleared.reset_index()
df_concepto_cleared.head()

,index,Concepto,concepto_cleared
0,0,COSTO DIRECTO DE OBRA,costo directo obra
1,0,ADMINISTRACION (22%),administracion
2,0,IMPROVISTOS (3%),improvistos
3,0,"UTILIDAD (4,2016806722%) MAS IVA 19%",utilidad mas iva
4,0,ESTUDIOS Y DISEÑOS MAS IVA 19%,estudios diseños mas iva


In [5]:
df_clear: pd.DataFrame = df_concepto_cleared[["concepto_cleared"]].drop_duplicates()
print(f"Cantidad original: {len(df_concepto_cleared):,.0f}")
print(f"Cantidad limpia: {len(df_clear):,.0f} => reducción del {100*(1 - len(df_clear)/len(df_concepto_cleared)):.2f}%")
df_clear.head()

Cantidad original: 131,337
Cantidad limpia: 23,115 => reducción del 82.40%


,concepto_cleared
0,costo directo obra
1,administracion
2,improvistos
3,utilidad mas iva
4,estudios diseños mas iva


In [6]:
INPUT_HOMOLOGA: str = "./resources/input/Tabla_homologación_2.xlsx"
df_homologa: pd.DataFrame = pd.read_excel(INPUT_HOMOLOGA)
df_homologa[~df_homologa["KeyWords"].isna()].head()

,Concepto,KeyWords
17,Honorarios y Comisiones,COMISION POR ADMINISTRACION
24,Contratos de construcción y urbanización,"COSTO DIRECTO, IMPREVISTOS, COSTOS DIRECTOS"
32,Cuota de admon,"CUOTA AIRE ACONDICIONADO, CUOTA EXPENSAS COMUN..."


In [7]:
df_keywords: pd.DataFrame = df_homologa[~df_homologa["KeyWords"].isna()].copy()
dict_keyword_concept: dict[str, str] = {}
for ix, row in df_keywords.iterrows():
    cleared_keywords: list[str] = [clean_concept(kw) for kw in row["KeyWords"].split(",") if len(clean_concept(kw)) > 0]
    for kw in cleared_keywords:
        dict_keyword_concept[kw] = row["Concepto"]
dict_keyword_concept

{'comision administracion': 'Honorarios y Comisiones',
 'costo directo': 'Contratos de construcción y urbanización',
 'imprevistos': 'Contratos de construcción y urbanización',
 'costos directos': 'Contratos de construcción y urbanización',
 'cuota aire acondicionado': 'Cuota de admon',
 'cuota expensas comunes': 'Cuota de admon',
 'cuota publicidad': 'Cuota de admon',
 'modulo aire acondicionado': 'Cuota de admon',
 'modulo gastos generales': 'Cuota de admon',
 'modulo publicidad': 'Cuota de admon',
 'expensas comunes': 'Cuota de admon',
 'modulo mercadeo publicidad': 'Cuota de admon'}

In [8]:
INPUT_HOMOLOGA: str = "./resources/input/Tabla_homologación_2.xlsx"
df_explicacion_tributario: pd.DataFrame = pd.read_excel(INPUT_HOMOLOGA, sheet_name="Hoja 4")
df_explicacion_tributario[["Concepto", "Descripción"]].head()

,Concepto,Descripción
0,Compras Generales,"Adquisición de bienes tangibles, materiales e ..."
1,Productos agrícolas/pecuarios (sin proceso ind...,Compra o suministro de productos del sector ag...
2,Café Pergamino o Cereza,Adquisición de café en estado de cereza (recié...
3,Combustibles derivados del petróleo,Compra o suministro de combustibles líquidos o...
4,Vehículos (Adquisición),"Compra de automotores, maquinaria rodante y de..."


In [9]:
from pandas._libs.missing import NAType
from typing import Union, Tuple, List, Optional

# Categoriza por KeyWord.
def fast_category(concept: str) -> Tuple[Union[str, NAType], Union[str, NAType]]:
    """
    Analiza un concepto y retorna tanto la categoría como el tipo de algoritmo.
    
    Retorna:
        (categoria, algoritmo)
    """
    # Validación inicial: longitud insuficiente o nulo
    # Se asume que 'concept' ya viene como string por el flujo previo
    if not isinstance(concept, str) or len(concept) <= 3:
        return "CONCEPTO VACIO", "NO-DATA"

    # Iteración sobre el diccionario de palabras clave
    for kw, cat in dict_keyword_concept.items():
        if kw == concept or kw in concept:
            return cat, "KEYWORD"

    # Caso por defecto: No hubo coincidencias
    return pd.NA, pd.NA

df_categorized: pd.DataFrame = df_clear.copy()
new_cols = ["categoria", "metodo_algoritmo"]
df_categorized[new_cols] = df_categorized["concepto_cleared"].apply(lambda x: pd.Series(fast_category(x)))
df_categorized[~df_categorized["categoria"].isna()].head()

,concepto_cleared,categoria,metodo_algoritmo
0,costo directo obra,Contratos de construcción y urbanización,KEYWORD
38,costo directo,Contratos de construcción y urbanización,KEYWORD
39,imprevistos,Contratos de construcción y urbanización,KEYWORD
45,,CONCEPTO VACIO,NO-DATA
161,costo directo acta parcial playa rica sede pri...,Contratos de construcción y urbanización,KEYWORD


In [10]:
df_process: pd.DataFrame = df_categorized.copy()
print(f"Cantidad de conceptos: {len(df_process):,.0f}, cantidad sin categorizar: {len(df_process[~df_process['categoria'].isna()]):,.0f}, faltante: {len(df_process) - len(df_process[~df_process['categoria'].isna()]):,.0f}")
print(f"Se esperan un total de {len(df_process)/20:,.0f} batchs")

Cantidad de conceptos: 23,115, cantidad sin categorizar: 274, faltante: 22,841
Se esperan un total de 1,156 batchs


In [11]:
CONCEPTOS_TRIBUTARIOS: List[str] = [f"{row.Concepto}: {row.Descripción}" for ix, row in df_explicacion_tributario[["Concepto", "Descripción"]].iterrows()]
CONCEPTOS_TRIBUTARIOS.append("DESCONOCIDO - No se logra entender o definir el concepto al que pertenece")
CONCEPTOS_TRIBUTARIOS[0:3]

['Compras Generales: Adquisición de bienes tangibles, materiales e insumos de uso general no clasificados en una subcategoría específica. Comprende mercancías, productos de consumo y suministros corporales susceptibles de inventariarse, almacenarse o transportarse físicamente.',
 'Productos agrícolas/pecuarios (sin proceso industrial): Compra o suministro de productos del sector agropecuario en su estado natural o primario, sin transformación industrial. Incluye frutas, verduras, cereales, carnes, lácteos y demás bienes de origen agrícola o pecuario que no han sido sometidos a proceso fabril.',
 'Café Pergamino o Cereza: Adquisición de café en estado de cereza (recién recolectado) o pergamino (con cáscara seca), sin proceso de trilla ni transformación industrial. Corresponde a la compra directa del grano en su forma primaria dentro de la cadena productiva cafetera.']

In [26]:
class ClasificacionItem(TypedDict):
    concept_ix: int  # Posición del concepto
    concepto: str    # Concepto original (espejo del input para verificar integridad)
    categoria: str   # Uno de los valores de CONCEPTOS_TRIBUTARIOS
    explicacion: str # Justificación breve (máx ~40 palabras)

def build_system_prompt(conceptos_tributarios: List[str]) -> str:
    opciones = "\n".join(f"- {c}" for c in conceptos_tributarios)
    return f"""Eres un experto en tributación colombiana especializado en retención en la fuente y clasificación de conceptos de facturación.

Tu única tarea es recibir una lista numerada de conceptos de factura y clasificar cada uno en exactamente uno de los siguientes conceptos tributarios:

{opciones}

Reglas de clasificación:
- "Empresas de Servicios Temporales" aplica cuando el concepto describe suministro, provisión o gestión de personal temporal, nómina externa o mano de obra en misión.
- "Honorarios y Comisiones" aplica a personas naturales o jurídicas que prestan servicios profesionales independientes sin relación laboral.
- "Consultoría General/Administración Delegada (PJ)" aplica cuando una persona jurídica presta asesoría, consultoría o administración de procesos.
- "Servicios Generales" es el comodín para servicios que no encajan claramente en ninguna categoría específica.
- "DESCONOCIDO" solo se usa cuando el concepto es completamente ambiguo o no corresponde a ninguna categoría.

Devuelve SIEMPRE un array JSON con exactamente tantos objetos como conceptos recibiste, en el MISMO ORDEN, con esta estructura:
[
  {{
    "concept_ix": <índice del concepto, se inicia en 0>,
    "concepto": "<texto original del concepto tal como fue recibido>",
    "categoria": "<categoría clasificada>",
    "explicacion": "<justificación en máximo 40 palabras>"
  }},
  ...
]

Sin texto adicional fuera del JSON.
"""

In [13]:
class CacheManager:
    """Mantiene una caché de system prompt activa y la renueva antes de que expire."""
    
    def __init__(self, conceptos_tributarios: List[str]):
        self._conceptos = conceptos_tributarios
        self._name: str | None = None
        self._expire_time: datetime | None = None

    @property
    def _is_valid(self) -> bool:
        if not self._name or not self._expire_time:
            return False
        remaining = (self._expire_time - datetime.now(timezone.utc)).total_seconds()
        return remaining > CACHE_SAFETY_MARGIN

    def _create(self) -> None:
        prompt = build_system_prompt(self._conceptos)
        cached = client.caches.create(
            model=MODEL,
            config=types.CreateCachedContentConfig(
                system_instruction=prompt,
                ttl=CACHE_TTL,
            ),
        )
        self._name: Optional[str] = cached.name
        self._expire_time = cached.expire_time
        print(f"Caché creada: name={self._name}  expira={self._expire_time}")

    def get(self) -> Optional[str]:
        """Devuelve un nombre de caché válido, creando o renovando si es necesario."""
        if not self._is_valid:
            self._create()
        return self._name

    def invalidate(self) -> Optional[str]:
        """Fuerza la recreación de la caché (llamar si la API reporta caché inválida)."""
        print("Caché inválida según la API; recreando…")
        self._name = None
        return self.get()

In [14]:
def _call_api(batch: List[str], cache_name: str) -> List[ClasificacionItem]:
    """Clasifica un batch de conceptos en una sola llamada."""
    numbered: str = "\n".join(f"{i}. {c}" for i, c in enumerate(batch))
    contents: types.ContentListUnionDict = [
        types.Content(role="user", parts=[types.Part.from_text(text=numbered)])
    ]
    config: types.GenerateContentConfig = types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(thinking_level=types.ThinkingLevel.HIGH),
        cached_content=cache_name,
        response_mime_type="application/json",
        response_schema=list[ClasificacionItem],
    )
    response = client.models.generate_content(model=MODEL, contents=contents, config=config)
    if response.text is None:
        raise ValueError("La API no generó respuesta.")
    return json.loads(response.text)

In [28]:
def _make_na_item(ix: int, concepto: str) -> ClasificacionItem:
    """Genera un item vacío con valores NA para un concepto sin clasificación."""
    return ClasificacionItem(
        concept_ix=ix,
        concepto=concepto,
        categoria="NA",
        explicacion="NA",
    )

def _align_results(
    batch: List[str],
    resultado: List[ClasificacionItem],
) -> List[ClasificacionItem]:
    """
    Alinea `resultado` con el orden original de `batch`.

    - Empareja por concepto (strip), con fallback a concept_ix.
    - Rellena con items NA los conceptos sin clasificación.
    - Corrige concept_ix e inyecta el concepto original en cada item.
    """
    # Índice por concepto normalizado → primer item que lo contenga
    by_concepto: dict[str, ClasificacionItem] = {}
    by_ix: dict[int, ClasificacionItem] = {}
    for item in resultado:
        key = item["concepto"].strip()
        by_concepto.setdefault(key, item)
        by_ix.setdefault(item["concept_ix"], item)

    aligned: List[ClasificacionItem] = []
    for ix, orig in enumerate(batch):
        norm = orig.strip()
        item = by_concepto.get(norm) or by_ix.get(ix)

        if item is None:
            print(f"Concepto en posición {ix} [{orig}] ausente en resultado; insertando NA.")
            aligned.append(_make_na_item(ix, orig))
        else:
            if item["concepto"].strip() != norm:
                print(
                    f"Concepto devuelto [{item['concepto']}] no coincide con original "
                    f"[{orig}]; corrigiendo."
                )
            # Normalizar siempre el concepto y su posición
            item["concepto"] = orig
            item["concept_ix"] = ix
            aligned.append(item)

    return aligned

def categorizar_batch(batch: List[str], cache_mgr: CacheManager, retries: int = 3) -> Tuple[List[ClasificacionItem], List[ClasificacionItem]]:
    """
    Clasifica un batch de hasta BATCH_SIZE conceptos con reintentos ante errores de caché.

    Args:
        batch:      Lista de textos de conceptos (máx BATCH_SIZE).
        cache_mgr:  Instancia de CacheManager para obtener/renovar la caché.
        retries:    Número máximo de reintentos ante fallos recuperables.
    """
    for attempt in range(1, retries + 1):
        resultado: List[ClasificacionItem] = []
        try:
            cache_name: str = cache_mgr.get() # pyright: ignore[reportAssignmentType]
            resultado: List[ClasificacionItem] = _call_api(batch, cache_name)
            if len(resultado) != len(batch):
                print(
                    f"El modelo devolvió {len(resultado)} items para un batch de {len(batch)}; "
                    "alineando y rellenando faltantes con NA."
                )

            return resultado, _align_results(batch, resultado)
        except Exception as exc:
            msg: str = str(exc).lower()
            is_cache_error: bool = "cache" in msg or "not found" in msg or "expired" in msg
            if is_cache_error and attempt < retries:
                cache_mgr.invalidate()
                continue
            if attempt < retries:
                wait: int = 2 ** attempt
                print(f"Intento {attempt}/{retries} falló ({exc}); reintentando en {wait}…")
                time.sleep(wait)
            else:
                print(f"Batch falló tras {retries} intentos: {exc}")
                return resultado, [_make_na_item(ix, orig) for ix, orig in enumerate(batch)]
    raise

In [34]:
from pathlib import Path

from tqdm.notebook import tqdm

CHECKPOINT_FILE: str = "./resources/output/fast_checkpoint.csv"

df_result: pd.DataFrame = df_process.copy()
df_result["explicacion"] = pd.NA

if os.path.exists(CHECKPOINT_FILE):
    print("Checkpoint encontrado. Cargando progreso previo...")
    df_checkpoint: pd.DataFrame = pd.read_csv(CHECKPOINT_FILE)
    cols_to_drop = [c for c in ["categoria", "metodo_algoritmo", "explicacion"] if c in df_result.columns]
    df_result: pd.DataFrame = df_result.drop(columns=cols_to_drop)
    df_result: pd.DataFrame = df_result.merge(df_checkpoint[["concepto_cleared", "categoria", "metodo_algoritmo", "explicacion"]], on="concepto_cleared", how="left")
else:
    print("Checkpoint no encontrado. Iniciando el proceso desde Dataframe original.")

def save_checkpoint(df: pd.DataFrame):
    df.to_csv(CHECKPOINT_FILE, index=False)

pending: int = df_result["categoria"].isna().sum()
total: int = len(df_result)
print(f"Pendientes: {pending}/{total}")

cache_mgr: CacheManager = CacheManager(CONCEPTOS_TRIBUTARIOS)
session_id: str = datetime.now().isoformat()
print("Session_id:", session_id)

# Índices de filas pendientes
pending_indices: List = df_result.index[df_result["categoria"].isna()].tolist()

with tqdm(total=total, initial=total - pending, unit="bloque") as pbar:
    for batch_start in range(0, len(pending_indices), BATCH_SIZE):
        batch_idx = pending_indices[batch_start : batch_start + BATCH_SIZE]
        batch_rows = df_result.loc[batch_idx]

        original = []
        conceptos: List[str] = [str(r) for r in batch_rows["concepto_cleared"]]
        #results: List[ClasificacionItem] = [ClasificacionItem(concept_ix=-1, concepto="...", categoria="...", explicacion="...")] * len(conceptos)
        original, results = categorizar_batch(conceptos, cache_mgr)
        
        # Actualizar el DataFrame con los resultados del batch
        for idx, item in zip(batch_idx, results):
            df_result.at[idx, "categoria"] = item["categoria"]
            df_result.at[idx, "metodo_algoritmo"] = MODEL
            df_result.at[idx, "explicacion"] = item["explicacion"]

        pbar.update(len(conceptos))
        pbar.set_postfix({"último": conceptos[-1][:30]})
        save_checkpoint(df_result)

        backup_answer: dict[str, List[ClasificacionItem] | List[str]] = {
            "input": conceptos,
            "output_raw": original,
            "output": results
        }
        path_backup = Path(f"./resources/output/backup_llm_answer/{session_id}")
        file_backup = path_backup / f"backup_{batch_start}_{batch_start + BATCH_SIZE}.json"
        path_backup.mkdir(parents=True, exist_ok=True)
        with open(file_backup, "w", encoding="utf-8") as f:
            f.write(json.dumps(backup_answer))
        #break

df_result.head()

Checkpoint encontrado. Cargando progreso previo...
Pendientes: 16/23115
Session_id: 2026-03-11T03:22:27.242430


100%|#########9| 23099/23115 [00:00<?, ?bloque/s]

Caché creada: name=cachedContents/2yr4re2xpmyghzgoxuup5hkfxq4jyqywyun8z8am  expira=2026-03-11 04:22:27.242081+00:00


,concepto_cleared,categoria,metodo_algoritmo,explicacion
0,costo directo obra,Contratos de construcción y urbanización,KEYWORD,NaN
1,administracion,Servicios Generales,gemini-3.1-flash-lite-preview,Concepto genérico sin descripción suficiente d...
2,improvistos,Servicios Generales,gemini-3.1-flash-lite-preview,Concepto contable genérico sin actividad defin...
3,utilidad mas iva,Servicios Generales,gemini-3.1-flash-lite-preview,Componente de un contrato sin detalle de la ac...
4,estudios diseños mas iva,Consultoría General/Administración Delegada (PJ),gemini-3.1-flash-lite-preview,Los estudios y diseños corresponden a labores ...
